# 내 구글 활동 시계열 분석 (My Activity Lens)

Google Takeout으로 받은 **Google 검색 기록**, **YouTube 시청·검색 기록**, **Chrome 방문 기록**을 시계열로 분석한다.
분석 질문과 정제 규칙은 [prd.md](prd.md) 5장과 4.4절을 따른다.

- 실행 전 가상환경(`.venv`)을 활성화하고, 노트북 커널도 `.venv`의 Python을 선택한다.
- 데이터 소스(`DATA_MODE`, 기본값 `raw`)
  - `raw`: `data/raw/`의 실제 데이터만. **리포트(REPORT.md)는 이 모드로 만든다** (PRD 1.4)
  - `auto`: 서비스별로 실제 데이터가 있으면 실제, 없으면 `data/sample/`의 합성 데이터
  - `sample`: 합성 데이터만
  - `raw`인데 실제 데이터가 없으면(저장소를 새로 받은 경우) 자동으로 `sample`로 바꾼다
- 샘플 데이터로 그린 서비스는 표와 그래프에 **"(샘플)"**로 표시한다.
- 검색 기록 실제 데이터 분석은 범위에서 제외했다 (PRD 1.4).
- 그래프는 `images/`에 저장되어 `REPORT.md`에 쓰인다.

## 1. 설정

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import features as F
from src import privacy, viz
from src.datasource import load_dataset
from src.profiles import channel_overlap

DATA_MODE = "raw"       # "raw" / "auto" / "sample"
EXCLUDE_DOMAINS = []    # 웹 방문 분석에서 뺄 도메인 (하위 도메인 포함)
SAVE_IMAGES = True

# 생활 단계 (이름, 시작일). 사용자 제공 일정 기준이며, 경계일은 "3월 말 퇴사", "4월 말 교육 시작"을 날짜로 옮긴 추정값이다
PHASES = [
    ("재직", "2025-01-01"),
    ("퇴사 후", "2026-04-01"),
    ("AI 교육", "2026-04-27"),
    ("AI 과정", "2026-07-01"),
]

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
print("한글 폰트:", viz.setup_style())


def show(fig, name):
    if SAVE_IMAGES:
        print("저장:", viz.save(fig, name).relative_to(ROOT))
    plt.show()

## 2. 데이터 로드와 기본 정보

In [ ]:
ds = load_dataset(DATA_MODE, exclude_domains=EXCLUDE_DOMAINS)
if ds.events.empty and DATA_MODE == "raw":
    print("data/raw/에 실제 데이터가 없어 샘플 데이터로 실행합니다. 리포트 이미지(images/)는 덮어쓰지 않습니다.")
    DATA_MODE = "sample"
    SAVE_IMAGES = False  # 리포트 그림은 실제 데이터로 만든 것이므로 샘플 결과로 바꾸지 않는다
    ds = load_dataset(DATA_MODE, exclude_domains=EXCLUDE_DOMAINS)
ev = ds.events
print("데이터 모드:", DATA_MODE)
print("서비스별 데이터 출처:", ds.sources)
for name, res in ds.results.items():
    print(f"\n[{name}] 프로필:", [(p.label, p.method) for p in res.profiles], "/ 병합 중복 제거:", res.duplicates_removed)
    for export, st in res.stats.items():
        print("  ", export, st)
    for w in res.warnings:
        print("   경고:", w)

### 실제 데이터의 이름 가리기 (NFR-03, NFR-10)

- 실제 웹 방문 도메인과 YouTube 채널은 `doc/domain_review.md`, `doc/channel_review.md`(로컬 전용, 커밋되지 않음)에서 체크한 것만 실명으로 표시한다.
- 검토 전에는 "도메인 1, 채널 1…"처럼 빈도 순위로 가린다. 검토 후에는 체크하지 않은 이름을 "기타 도메인", "기타 채널"로 묶는다.
- `label` 컬럼: 검색어·도메인·채널(YouTube 시청)을 가린 이름. 그래프와 표는 이 컬럼을 쓴다.
- 실제 검색어는 "검색어 1, 2…"로 가린다. 샘플 데이터는 가리지 않는다.

In [ ]:
ev["label"] = ev["text"].astype("object")
is_watch = ev["service"] == "yt_watch"
ev.loc[is_watch, "label"] = ev.loc[is_watch, "channel"]  # YouTube 시청은 콘텐츠 채널 기준으로 집계
raw_watch = is_watch & (ev["source"] == "raw")
raw_web = (ev["service"] == "web_visit") & (ev["source"] == "raw")
raw_query = ev["service"].isin(["search", "yt_search"]) & (ev["source"] == "raw")
public_domains = privacy.read_public_names()
if raw_web.any():
    print("도메인 검토 목록:", privacy.write_review(ev.loc[raw_web, "domain"]).relative_to(ROOT))
    ev.loc[raw_web, "label"] = privacy.mask_labels(ev.loc[raw_web, "domain"], public_domains)
if raw_query.any():
    ev.loc[raw_query, "label"] = privacy.mask_labels(ev.loc[raw_query, "text"], None, kind="검색어")
public_channels = privacy.read_public_names(privacy.CHANNEL_REVIEW_PATH)
if raw_watch.any():
    print("채널 검토 목록:", privacy.write_review(ev.loc[raw_watch, "channel"], privacy.CHANNEL_REVIEW_PATH,
                                                kind="채널").relative_to(ROOT))
    ev.loc[raw_watch, "label"] = privacy.mask_labels(ev.loc[raw_watch, "channel"], public_channels, kind="채널")
print("공개 도메인:", "검토 전 (모두 가림)" if public_domains is None else sorted(public_domains))
print("공개 채널:", "검토 전 (모두 가림)" if public_channels is None else sorted(public_channels))

In [ ]:
series = F.daily_by_service(ev)
gaps = {s: F.find_gaps(x) for s, x in series.items()}
outliers = {s: F.flag_outliers(x) for s, x in series.items()}
label = lambda s: viz.service_label(s, ds.sources)

summary = pd.DataFrame({
    label(s): {
        "출처": ds.sources[s], "시작": x.index.min().date(), "종료": x.index.max().date(),
        "달력 일수": len(x), "활동 있는 날": int((x > 0).sum()), "총 건수": int(x.sum()),
        "일평균(달력 기준)": round(x.mean(), 1),
        "중단 의심 구간": len(gaps[s]), "중단 일수": int(gaps[s]["days"].sum()) if len(gaps[s]) else 0,
        "이상치 일수": int(outliers[s].sum()),
    }
    for s, x in series.items()
}).T
summary

### 결측·이상치 처리 기준 (PRD 4.4)

- **활동이 없는 날**은 결측이 아니라 실제 0으로 채운다. 단, 0이 7일 넘게 이어지면 "기록 중단 의심 구간"으로 따로 표시한다.
- **이상치**는 일별 건수가 `Q3 + 3×IQR`을 넘는 날이다. 몰아보기 같은 실제 행동일 수 있으므로 **표시만 하고 제거하지 않는다**.
- 광고 시청, 삭제된 영상, 중복 레코드, 로컬·내부 주소, 1분 안 반복 방문은 파싱 단계에서 처리했다(위 파싱 통계 참고).

In [ ]:
show(viz.plot_outliers_gaps(series, outliers, gaps, ds.sources, phases=PHASES), "04_outliers_gaps.png")
for s, g in gaps.items():
    if len(g):
        print(label(s), "기록 중단 의심 구간")
        print(g.assign(start=g["start"].dt.date, end=g["end"].dt.date).to_string(index=False), "\n")

## 3. Q1 추세: 활동량은 늘었나, 줄었나?

In [ ]:
show(viz.plot_daily_trend(series, ds.sources, phases=PHASES), "01_daily_trend_ma.png")

trend = pd.DataFrame({
    label(s): {"첫 8주 일평균": x.iloc[:56].mean(), "마지막 8주 일평균": x.iloc[-56:].mean()}
    for s, x in series.items()
}).T
trend["변화율(%)"] = (trend["마지막 8주 일평균"] / trend["첫 8주 일평균"] - 1) * 100
trend.round(1)

In [ ]:
monthly = pd.DataFrame({label(s): F.resample(x, "M") for s, x in series.items()})
monthly.index = monthly.index.strftime("%Y-%m")
monthly

In [ ]:
# 서비스끼리 같은 방향으로 움직이나: 주간 합계의 상관계수 (공통 기간)
weekly = pd.DataFrame({label(s): F.resample(x, "W") for s, x in series.items()}).dropna()
weekly.corr().round(2)

In [ ]:
weekly_pct = {s: F.pct_change(F.resample(x, "W")) for s, x in series.items()}
show(viz.plot_weekly_change(weekly_pct, ds.sources), "02_weekly_change.png")
pd.DataFrame({label(s): {"변화율 중앙값(절댓값 %)": p.abs().median(), "최대 증가(%)": p.max(), "최대 감소(%)": p.min()}
              for s, p in weekly_pct.items()}).T.round(1)

## 4. Q2 주기성: 요일·시간대 패턴이 있나?

In [ ]:
matrices = {str(s): F.weekday_hour_matrix(g) for s, g in ev.groupby("service", observed=True)}
show(viz.plot_heatmaps(matrices, ds.sources), "03_weekday_hour_heatmap.png")

rhythm = {}
for s, g in ev.groupby("service", observed=True):
    x, hours = series[str(s)], g["ts_kst"].dt.hour
    weekday, weekend = x[x.index.weekday < 5].mean(), x[x.index.weekday >= 5].mean()
    rhythm[label(str(s))] = {
        "평일 일평균": weekday, "주말 일평균": weekend, "주말/평일(배)": weekend / weekday,
        "피크 시간": int(hours.mode()[0]),
        "오전(6~11시) %": hours.between(6, 11).mean() * 100,
        "오후(12~17시) %": hours.between(12, 17).mean() * 100,
        "저녁(18~23시) %": hours.between(18, 23).mean() * 100,
        "심야(0~5시) %": hours.between(0, 5).mean() * 100,
    }
pd.DataFrame(rhythm).T.round(2)

## 5. Q3 급변 구간: 언제 튀었고, 그날 무엇이 많았나?

In [ ]:
rows = []
for s, x in series.items():
    dates = ev["ts_kst"].dt.tz_localize(None).dt.normalize()
    for d in x.index[outliers[s]]:
        top = ev.loc[(ev["service"] == s) & (dates == d), "label"].value_counts().head(3)
        rows.append({
            "서비스": label(s), "날짜": d.date(), "요일": F.WEEKDAYS[d.weekday()], "건수": int(x[d]),
            "활동일 중앙값 대비(배)": round(x[d] / x[x > 0].median(), 1),
            "상위 항목": ", ".join(f"{k}({v})" for k, v in top.items()),
        })
pd.DataFrame(rows)

## 6. Q4 관심사 변화: 분기별 상위 항목

In [ ]:
panels = []
if "search" in series:
    panels.append((label("search") + " 검색어", ev[(ev["service"] == "search") & (ev["activity"] == "searched")], "label"))
if "yt_watch" in series:
    panels.append((label("yt_watch") + " 채널", ev[ev["service"] == "yt_watch"], "label"))
if "web_visit" in series:
    panels.append((label("web_visit") + " 도메인", ev[ev["service"] == "web_visit"], "label"))
show(viz.plot_top_by_quarter(panels, n=5), "05_top_items_by_quarter.png")

## 7. Q6 채널 비교: 내 채널마다 시청 패턴이 다른가?

In [ ]:
watch = ev[ev["service"] == "yt_watch"]
if watch["profile"].nunique() >= 2:
    show(viz.plot_profile_compare(watch), "06_profile_compare.png")
    display(channel_overlap(ev))
    display(watch.groupby("profile", observed=True)["ts_kst"].agg(
        건수="size", 피크시간=lambda t: int(t.dt.hour.mode()[0]),
        주말비율=lambda t: round((t.dt.weekday >= 5).mean() * 100, 1)))
else:
    print("프로필이 1개라 비교를 건너뜁니다.")

## 8. Q7 웹 사용 패턴

In [ ]:
web = ev[ev["service"] == "web_visit"]
if len(web):
    suffix = "" if ds.sources["web_visit"] == "sample" else (
        " (도메인 이름 가림)" if public_domains is None else " (공개 도메인만 실명)")
    show(viz.plot_domain_trend(web, "label", k=6, title_suffix=suffix, phases=PHASES), "07_web_domains_trend.png")
    top = web["label"].value_counts().head(10)
    display(pd.DataFrame({"방문 수": top, "비율(%)": (top / len(web) * 100).round(1)}))

In [ ]:
if len(series) >= 2:
    show(viz.plot_combined_activity(series, ds.sources, phases=PHASES), "08_combined_activity.png")
    start = max(x.index.min() for x in series.values())
    end = min(x.index.max() for x in series.values())
    overlap = pd.DataFrame({label(s): x.loc[start:end] for s, x in series.items()})
    display((overlap.sum() / overlap.to_numpy().sum() * 100).round(1).rename("공통 기간 비율(%)").to_frame())

## 9. Q8 생활 단계 비교: 재직 → 퇴사 후 → AI 교육 → AI 과정

In [ ]:
show(viz.plot_phase_compare(ev, series, PHASES, ds.sources), "09_phase_compare.png")
for s, x in series.items():
    print(label(s))
    display(F.phase_summary(ev[ev["service"] == s], x, PHASES).round(1))

In [ ]:
# 단계별 상위 웹 도메인 (가린 이름) 과 상위 5개 집중도
web = ev[ev["service"] == "web_visit"]
if len(web):
    web_phase = F.assign_phase(web["ts_kst"], PHASES).to_numpy()
    rows = {}
    for name, _ in PHASES:
        top = web.loc[web_phase == name, "label"].value_counts()
        if len(top):
            rows[name] = {"방문 수": int(top.sum()), "고유 도메인": len(top),
                          "상위 5개 비율(%)": round(top.head(5).sum() / top.sum() * 100, 1),
                          "상위 5개": ", ".join(f"{k}({v})" for k, v in top.head(5).items())}
    display(pd.DataFrame(rows).T)

## 10. 다음 단계

- M4: 위 표와 그래프의 수치를 근거로 인사이트를 3개 이상 쓴다. 관찰(Fact)과 해석(가설)을 나눠 쓴다.
- `doc/domain_review.md`, `doc/channel_review.md`에서 공개할 이름을 체크하면, 다시 실행할 때 그래프에 실명이 나온다.